In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "randomPassword2025"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
try:
    image_filename = 'code_files/Grazioso Salvare Logo.png' # replace with your own image
    with open(image_filename, 'rb') as f:
        encoded_image = base64.b64encode(f.read()).decode()
    logo_component = html.Img(src='data:image/png;base64,{}'.format(encoded_image), style={'height': '75px'})
except FileNotFoundError:

    logo_component = html.H3("Grazioso Salvare Dashboard - Marshon Hughes")

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    logo_component,
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),
    html.Div([
        
#FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
        html.Label("Rescue Filter"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {"label": "Water Rescue", "value" : "Water Rescue"},
                {"label" : "Mountain/Wilderness Rescue" , "value": "Mountain/Wilderness Rescue"},
                {"label" : "Disaster/Individual Tracking", "value" : "Disaster/Individual Tracking"},
                {"label" : "Reset All Values", "value" : "Reset"},
            ],
            value="Reset",
            labelStyle={"display":"block"}
        ),
    ]),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 
                       
                        
                        sort_action = "native",
                        filter_action = "native",
                        page_size = 10,
                        row_selectable='single',
                        selected_rows=[0],
                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
## FIX ME Add code to filter interactive data table with MongoDB queries

#

#        
#        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
#        data=df.to_dict('records')
#       
#       
#        return (data,columns)

# Display the breeds of animal based on quantity represented in
# the data table
    
    dff = df.copy()
    BREED_COLUMN = 'breed' if 'breed' in dff.columns else None
    SEX_COLUMN = 'sex_upon_outcome' if 'sex_upon_outcome' in dff.columns else None
    AGE_WEEKS = 'age_upon_outcome_in_weeks' if 'age_upon_outcome_in_weeks' in dff.columns else None
    ANIMAL_TYPE = 'animal_type' if 'animal_type' in dff.columns else None 
    
    if filter_type == "Water Rescue":
        if BREED_COLUMN:
            dff = dff[dff[BREED_COLUMN].isin(["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"])]
        if SEX_COLUMN: 
            dff = dff[dff[SEX_COLUMN]== "Intact Female"]
        if AGE_WEEKS: 
            dff = dff[(dff[AGE_WEEKS] >= 26) & (dff[AGE_WEEKS] <= 156)]
        if ANIMAL_TYPE: 
            dff = dff[dff[ANIMAL_TYPE] == "Dog"]
    
    elif filter_type == "Mountain/Wilderness Rescue":
        if BREED_COLUMN: 
            dff = dff[dff[BREED_COLUMN].isin(["German Shepherd", "Alaskan Malamute", "Old English Sheepdog" , "Siberian Husky", "Rottweiler"])]
        if SEX_COLUMN: 
            dff = dff[dff[SEX_COLUMN]=="Intact Male"]
        if AGE_WEEKS: 
            dff = dff[(dff[AGE_WEEKS] >= 26) & (dff[AGE_WEEKS] <= 156)]
        if ANIMAL_TYPE: 
            dff = dff[dff[ANIMAL_TYPE] == "Dog"]
    
    elif filter_type == "Disaster/Individual Tracking":
        if BREED_COLUMN: 
            dff = dff[dff[BREED_COLUMN].isin(["Doberman Pinscher", "German Shepherd", "Golden Retriever" , "Bloodhound", "Rottweiler", "Labrador Retriever", "Coonhound"])]
        if SEX_COLUMN: 
            dff = dff[dff[SEX_COLUMN]== "Intact Male"]
        if AGE_WEEKS: 
            dff = dff[(dff[AGE_WEEKS] >= 20) & (dff[AGE_WEEKS] <= 300)]
        if ANIMAL_TYPE: 
            dff = dff[dff[ANIMAL_TYPE] == "Dog"]
    return dff.to_dict('records')                                                 
    
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #

    #return [
    #    dcc.Graph(            
    #        figure = px.pie(df, names='breed', title='Preferred Animals')
    #    )    
    #]
    dff = pd.DataFrame(viewData) if viewData else  df
    col = 'breed' if 'breed' in dff.columns else dff.columns[0]
    if len(dff) == 0:
        return dcc.Graph(figure=px.pie(pd.DataFrame({col: [], "count": []}), names = col, title = "Breed Distribution"))
    return dcc.Graph(figure=px.pie(dff, names=col, title="Breed Distribution"))
                                                       
                                                       
                                                       
    
#This callback will highlight a cell on the data table when the user selects it
                                                       
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    selected_columns = selected_columns or []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(port=8055) 

Documents found
Dash app running on https://barcodejoshua-icebergfast-3000.codio.io/proxy/8055/
